### 03 Result Inspection
Load the collection summary, inspect metadata, and filter reconstruction results.

In [ ]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
sys.path.append(str(project_root / "src"))

#### Imports

In [ ]:
import ast
import pandas as pd
from IPython.display import display

#### Load Collection Summary

In [ ]:
collection_csv = project_root / "outputs" / "collection_summary.csv"
df = pd.read_csv(collection_csv, encoding="utf-8-sig")

print("Collection summary:", collection_csv)
print("Rows   :", len(df))
print("Columns:", len(df.columns))

df.head()

#### Basic Statistics

In [ ]:
print("Total files   :", len(df))

if "success" in df.columns:
    print("Success files :", int(df["success"].sum()))
    print("Failed files  :", int((~df["success"]).sum()))

if "freq_offset_hz" in df.columns:
    valid_foff = df.loc[df["success"] == True, "freq_offset_hz"].dropna()
    if len(valid_foff) > 0:
        print("Mean freq offset (Hz):", float(valid_foff.mean()))
        print("Std  freq offset (Hz):", float(valid_foff.std()))

#### Key Columns Preview

In [ ]:
cols_to_show = [
    "stem",
    "success",
    "run_mode",
    "input_batch_name",
    "loaded_samples",
    "resampled_samples",
    "available_frames",
    "freq_offset_hz",
    "Fs",
    "Fc",
    "capture_time_s",
    "total_overrun",
    "capture_meta.sample_id",
    "capture_meta.slide_index",
    "capture_meta.font_size_pt",
    "capture_meta.sdr_model",
    "capture_meta.antenna_model",
    "capture_meta.test_distance_cm",
    "capture_meta.environment",
    "stack_frame_start",
    "stack_frame_count",
    "stack_direction",
    "stack_image_mode",
]

existing_cols = [c for c in cols_to_show if c in df.columns]
display(df[existing_cols].head(20))

#### Show Batch Names

In [ ]:
if "input_batch_name" in df.columns:
    batch_names = (
        df["input_batch_name"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )
    display(batch_names.to_frame(name="batch_name"))
else:
    print("No input_batch_name column found.")

#### Success / Failure Split

In [ ]:
success_df = df[df["success"] == True].copy() if "success" in df.columns else df.copy()
fail_df = df[df["success"] == False].copy() if "success" in df.columns else pd.DataFrame()

print("Success rows:", len(success_df))
print("Fail rows   :", len(fail_df))

display(success_df.head(10))

if len(fail_df) > 0:
    display(fail_df.head(10))

#### Filter by Metadata

In [ ]:
filtered = success_df.copy()

# Example filters:
# filtered = filtered[filtered["input_batch_name"] == "output_16x9_20pt_pages_0001-0100_CF_742.500MHz_FS_61.440MSPS_20260402_131046"]
# filtered = filtered[filtered["capture_meta.font_size_pt"] == 20]
# filtered = filtered[filtered["capture_meta.slide_index"].between(1, 20)]
# filtered = filtered[filtered["total_overrun"] == 0]
# filtered = filtered[filtered["stack_frame_count"] == 4]

print("Filtered rows:", len(filtered))
display(filtered.head(20))

#### Sort and Inspect

In [ ]:
sort_cols = [c for c in [
    "input_batch_name",
    "capture_meta.sample_id",
    "capture_meta.slide_index",
    "freq_offset_hz"
] if c in filtered.columns]

if len(sort_cols) > 0:
    filtered_sorted = filtered.sort_values(sort_cols).copy()
else:
    filtered_sorted = filtered.copy()

display(filtered_sorted.head(30))

#### Quick Group Summary

In [ ]:
group_col = "capture_meta.font_size_pt"

if group_col in success_df.columns:
    summary_by_group = (
        success_df.groupby(group_col)
        .agg(
            count=("stem", "count"),
            mean_foff=("freq_offset_hz", "mean"),
            mean_overrun=("total_overrun", "mean"),
        )
        .reset_index()
    )
    display(summary_by_group)
else:
    print(f"Column not found: {group_col}")

#### Parse Saved Files

In [ ]:
def parse_saved_files(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return [x]
    return []

if "saved_files" in df.columns:
    df["saved_files_parsed"] = df["saved_files"].apply(parse_saved_files)
    display(df[["stem", "input_batch_name", "saved_files_parsed"]].head(5))
else:
    print("No saved_files column found.")

#### Lookup by Sample or Slide

In [ ]:
sample_id = None   # e.g. 37
slide_id = None    # e.g. 37
batch_name = None  # e.g. "output_16x9_20pt_pages_0001-0100_CF_742.500MHz_FS_61.440MSPS_20260402_131046"

lookup_df = df.copy()

if batch_name is not None and "input_batch_name" in lookup_df.columns:
    lookup_df = lookup_df[lookup_df["input_batch_name"] == batch_name]

if sample_id is not None and "capture_meta.sample_id" in lookup_df.columns:
    lookup_df = lookup_df[lookup_df["capture_meta.sample_id"] == sample_id]

if slide_id is not None and "capture_meta.slide_index" in lookup_df.columns:
    lookup_df = lookup_df[lookup_df["capture_meta.slide_index"] == slide_id]

display(lookup_df)

#### Preview One Sample

In [ ]:
row_idx = 0

if len(filtered_sorted) == 0:
    print("No rows available.")
else:
    row = filtered_sorted.iloc[row_idx]
    print("Batch     :", row.get("input_batch_name"))
    print("Stem      :", row.get("stem"))
    print("Sample ID :", row.get("capture_meta.sample_id"))
    print("Slide ID  :", row.get("capture_meta.slide_index"))

    saved = row.get("saved_files_parsed", [])
    for i, p in enumerate(saved, start=1):
        print(f"[{i}] {p}")

#### Export Filtered Result (Optional)

In [ ]:
export_csv = project_root / "outputs" / "filtered_selection.csv"
filtered_sorted.to_csv(export_csv, index=False, encoding="utf-8-sig")
print("Saved filtered CSV to:", export_csv)